# Milestone 04 : Model Comparison

### 1. Library Import and Data Loading

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score

# Load engineered data
df = pd.read_csv('data/engineered_data.csv')

In [ ]:
# Separate features (X) and target (y)
X = df.drop(columns=['price', 'date'], errors='ignore')
y = df['price']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 2. Retraining the Baseline Model

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

# Create a pipeline that imputes missing values before applying Linear Regression
baseline_model = make_pipeline(
    SimpleImputer(strategy='median'),
    LinearRegression()
)

# Train the model
baseline_model.fit(X_train, y_train)

# Evaluate scores
base_train_score = baseline_model.score(X_train, y_train)
base_test_score = baseline_model.score(X_test, y_test)

### 3. Tuning and Training Random Forest with GridSearchCV

In [ ]:
rf = RandomForestRegressor(random_state=42)
rf_param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10]
}

rf_grid = GridSearchCV(estimator=rf, param_grid=rf_param_grid, cv=3, scoring='r2', n_jobs=-1)
rf_grid.fit(X_train, y_train)

# Best estimator and scores
best_rf = rf_grid.best_estimator_
rf_train_score = best_rf.score(X_train, y_train)
rf_test_score = best_rf.score(X_test, y_test)
rf_best_params = rf_grid.best_params_

print("Best parameters for Random Forest:", rf_best_params)

Best parameters for Random Forest: {'max_depth': 10, 'n_estimators': 50}


### 4. Tuning and Training Gradient Boosting with GridSearchCV

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Create a pipeline with an Imputer and the Gradient Boosting model
gb_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('gb', GradientBoostingRegressor(random_state=42))
])

# Adjust parameter grid to match the Pipeline structure
gb_param_grid = {
    'gb__n_estimators': [50, 100],
    'gb__max_depth': [3, 5]
}

gb_grid = GridSearchCV(estimator=gb_pipeline, param_grid=gb_param_grid, cv=3, scoring='r2', n_jobs=-1)
gb_grid.fit(X_train, y_train)

# Best estimator and scores
best_gb = gb_grid.best_estimator_
gb_train_score = best_gb.score(X_train, y_train)
gb_test_score = best_gb.score(X_test, y_test)
gb_best_params = gb_grid.best_params_

print("Best parameters for Gradient Boosting:", gb_best_params)

Best parameters for Gradient Boosting: {'gb__max_depth': 5, 'gb__n_estimators': 100}


### 5. Creating the Comparison Table

In [ ]:
# Create the summary DataFrame
comparison_data = {
    "Model": ["Baseline", "Random Forest", "Gradient Boosting"],
    "Train Score": [f"{base_train_score:.4f}", f"{rf_train_score:.4f}", f"{gb_train_score:.4f}"],
    "Test Score": [f"{base_test_score:.4f}", f"{rf_test_score:.4f}", f"{gb_test_score:.4f}"],
    "Best Params": ["N/A", str(rf_best_params), str(gb_best_params)]
}

df_comparison = pd.DataFrame(comparison_data)
display(df_comparison)

,Model,Train Score,Test Score,Best Params
0,Baseline,0.9012,0.8973,N/A
1,Random Forest,0.9989,0.9937,"{'max_depth': 10, 'n_estimators': 50}"
2,Gradient Boosting,0.9993,0.9956,"{'gb__max_depth': 5, 'gb__n_estimators': 100}"


### Conclusion on Model Comparison

Based on the results obtained, we can draw the following conclusions:

1. **Superiority of Non-Linear (Ensemble) Models:**
   Both **Random Forest** ($R^2$ Test: ~0.9937) and **Gradient Boosting** ($R^2$ Test: ~0.9956) significantly outperform the **Baseline Linear Regression** model ($R^2$ Test: ~0.8973). This demonstrates that the relationships between real estate features and the purchase price are highly non-linear and involve complex interactions that only decision-tree-based architectures can capture effectively.

2. **Ultimate Performance:**
   **Gradient Boosting** is our best-performing model. In addition to delivering the highest test score (explaining 99.56% of the variance), it utilizes optimized hyperparameters (`max_depth: 5`, `n_estimators: 100`), making it highly robust.

3. **Overfitting Analysis:**
   - **Linear Regression:** No sign of overfitting (Train/Test gap of ~0.4%). The model is instead slightly underfitting compared to ensemble models.
   - **Random Forest:** We observe a nearly perfect training score (~99.89%) versus a very high but lower test score (~99.37%).
   - **Gradient Boosting:** This is the most balanced model. The gap between the training score (~99.93%) and the test score (~99.56%) is extremely small (~0.37%). The risk of overfitting is thus exceptionally well controlled thanks to the sequential learning process and optimal hyperparameters found via `GridSearchCV`.

In [11]:
import joblib
import os

# Assuming 'best_rf' or 'best_model' is your best performing model
joblib.dump(best_rf, 'model.pkl')
print("Model successfully saved as 'model.pkl'!")

Model successfully saved as 'model.pkl'!
